In [5]:
import pandas as pd
from collections import Counter

In [2]:
peo = pd.read_csv("peo.csv")
saf = pd.read_csv("saf.csv")
yos = pd.read_csv("yos.csv")

print(peo.columns.tolist())
print(saf.columns.tolist())
print(yos.columns.tolist())
print(peo.shape, saf.shape, yos.shape)

['annotation_id', 'annotator', 'author_id', 'collection_method', 'conversation_id', 'created_at', 'film', 'has_keywords', 'id', 'is_relevant_v2', 'lead_time', 'sentiment', 'to_keep', 'tweet', 'updated_at']
['annotation_id', 'annotator', 'author_id', 'collection_method', 'conversation_id', 'created_at', 'film', 'has_keywords', 'id', 'is_relevant_v2', 'lead_time', 'sentiment', 'to_keep', 'tweet', 'updated_at']
['annotation_id', 'annotator', 'author_id', 'collection_method', 'conversation_id', 'created_at', 'film', 'has_keywords', 'id', 'is_relevant_v2', 'lead_time', 'sentiment', 'to_keep', 'tweet', 'updated_at']
(4068, 15) (4068, 15) (4068, 15)


In [3]:
print(peo['id'].nunique(), saf['id'].nunique(), yos['id'].nunique())
print()
print(peo['sentiment'].value_counts())
print()
print(saf['sentiment'].value_counts())
print()
print(yos['sentiment'].value_counts())

4068 4068 4068

sentiment
delete     2853
LOVE        517
SADNESS     320
JOY         141
ANGER       127
FEAR        106
Name: count, dtype: int64

sentiment
LOVE       1363
SADNESS     761
JOY         735
ANGER       583
FEAR        361
delete      258
Name: count, dtype: int64

sentiment
delete     1392
SADNESS     667
LOVE        654
JOY         642
ANGER       513
FEAR        198
Name: count, dtype: int64


In [6]:
merged = peo[['id', 'tweet', 'sentiment']].rename(columns={'sentiment': 'sentiment_peo'})
merged = merged.merge(saf[['id', 'sentiment']].rename(columns={'sentiment': 'sentiment_saf'}), on='id')
merged = merged.merge(yos[['id', 'sentiment']].rename(columns={'sentiment': 'sentiment_yos'}), on='id')

for col in ['sentiment_peo', 'sentiment_saf', 'sentiment_yos']:
    merged[col] = merged[col].str.lower()

print(merged.shape)

(4068, 5)


In [7]:
def resolve_row(row):
    votes = [row['sentiment_peo'], row['sentiment_saf'], row['sentiment_yos']]
    counts = Counter(votes)
    most_common_label, most_common_count = counts.most_common(1)[0]
    
    if most_common_count >= 2:
        # ada majority vote (2 atau 3 dari 3 sepakat)
        return most_common_label, False  # not conflicting
    else:
        # semua vote beda (no majority)
        if 'delete' in votes:
            return 'delete', False  # delete menang walau no majority
        else:
            return None, True  # conflicting: 3 emosi beda, ga ada delete

results = merged.apply(lambda row: resolve_row(row), axis=1)
merged['final_label'] = results.apply(lambda x: x[0])
merged['is_conflicting'] = results.apply(lambda x: x[1])

n_conflicting = merged['is_conflicting'].sum()
n_resolved = (~merged['is_conflicting']).sum()

print(f"Conflicting (no majority, no delete): {n_conflicting}")
print(f"Resolved: {n_resolved}")
print(f"Total: {len(merged)}")

Conflicting (no majority, no delete): 50
Resolved: 4018
Total: 4068


In [10]:
# resolved: yang ada hasil akhirnya
resolved_df = merged[~merged['is_conflicting']].copy()
resolved_df = resolved_df[resolved_df['final_label'] != 'delete']  # buang yang delete
resolved_df = resolved_df[['id', 'tweet', 'final_label']].rename(columns={'final_label': 'label'})

def anonymize(text):
    return re.sub(r'@\w+', '[USERNAME]', text)

resolved_df['tweet'] = resolved_df['tweet'].apply(anonymize)

# cek hasil
print(resolved_df['tweet'].head(5).tolist())
print()

# conflicting: yang belum ada keputusan
conflicting_df = merged[merged['is_conflicting']].copy()
conflicting_df = conflicting_df[['id', 'tweet', 'sentiment_peo', 'sentiment_saf', 'sentiment_yos']]

print(f"Resolved (non-delete): {len(resolved_df)}")
print(f"Conflicting: {len(conflicting_df)}")
print()
print(resolved_df['label'].value_counts())

resolved_df.to_csv("test_dataset.csv", index=False)
conflicting_df.to_csv("conflicting_tweets.csv", index=False)
print("\nSaved: test_dataset.csv, conflicting_tweets.csv")

['habis nonton film sore istri dari masa depan ternyata capek banget yah, ini tuh capek-nya ngaruh ke mental gua banget fakk', 'baru nonton film SORE , dan emang masterpiece-nya itu di endingnya 🥲  GILAAK NI FILM', 'Yeyy gak sabar mau rewatch lagi! Film sore istri masa depan next yah ditunggu 😍', '[USERNAME] Daripada film perang kota mending ini bangetttttt', 'Film perang kota tuh ceritanya ttg apa sih? Kok fokusnya ke adegan panas cinta segitiga aja?']

Resolved (non-delete): 2029
Conflicting: 50

label
love       673
sadness    532
anger      410
joy        250
fear       164
Name: count, dtype: int64

Saved: test_dataset.csv, conflicting_tweets.csv


In [9]:
import re

# cek beberapa contoh tweet yang ada mention
sample = resolved_df[resolved_df['tweet'].str.contains(r'@\w+', na=False)]['tweet'].head(5)
for t in sample:
    print(t)
    print('---')

@CenayangFilm Daripada film perang kota mending ini bangetttttt
---
@pillofpressure iih btw salfok ibu produser atau apa itu dr film jumbo, senyum pas liat raisa. yuk bisa yuk bu angkut film sepaket ☺️
---
@Adriandhy Yeeeayy akhirnya anak2 ku bisa nonton film Jumbo 🥹🫶🏻
---
@IndoPopBase Yeyyy bisa rewatch film Jumbo sambil rebahan di kamar!🤗
---
@basebuku Jadi kebayang Reza Rahadian yang ngomong ngapak kaya di film Gowok.
---


# Legacy

In [12]:
agreed = both_keep[both_keep['sentiment_peo'] == both_keep['sentiment_saf']].copy()
agreed = agreed[['id', 'tweet', 'sentiment_peo']].rename(columns={'sentiment_peo': 'label'})
agreed['label'] = agreed['label'].str.lower()

print(agreed['label'].value_counts())
print(f"\nTotal: {len(agreed)}")

label
love       376
sadness    237
anger       86
fear        70
joy         69
Name: count, dtype: int64

Total: 838


In [13]:
agreed.to_csv("test_dataset.csv", index=False)
print("Saved: test_dataset.csv")

Saved: test_dataset.csv


In [2]:
test_df = pd.read_csv("test_dataset.csv")
print(test_df['label'].unique())

<ArrowStringArray>
['sadness', 'love', 'joy', 'anger', 'fear']
Length: 5, dtype: str
